# NASA C-MAPSS FD001 | Turbofan Remaining Useful Life Prediction

**Reproducible baseline study · Linear Regression and Random Forest · Engine-disjoint validation**

This notebook investigates remaining useful life (RUL) prediction on the **simulated NASA C-MAPSS FD001** turbofan dataset. We compare instantaneous sensor baselines with short-history features and evaluate the selected model on the official test endpoints. We examine not only absolute prediction error but also **RUL overestimation**, which may matter in maintenance planning.

**Protocol:** Select model features on engine-disjoint validation data; freeze the six-feature Random Forest configuration; retrain on all 100 training engines; evaluate once on the official 100-engine test set. The official test set is not used for model selection.

**Dataset:** Kaggle mirror `behrad3d/nasa-cmaps`, containing NASA C-MAPSS FD001 files. Attach the dataset to this notebook before running it.

## 1. Imports, configuration and dataset loading

Each FD001 row contains an engine identifier, operating cycle, three operating settings, and 21 sensor measurements. Training engines run to failure; test engines stop at an earlier observation point. The separate `RUL_FD001.txt` file contains each test engine's true RUL **at its last observed cycle**.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_DIR = Path("/kaggle/input/datasets/behrad3d/nasa-cmaps/CMaps")
assert DATA_DIR.is_dir(), f"Dataset folder not found: {DATA_DIR}. Attach the nasa-cmaps dataset in Kaggle."

columns = (["engine_id", "cycle"]
           + [f"setting_{i}" for i in range(1, 4)]
           + [f"sensor_{i}" for i in range(1, 22)])
train_raw = pd.read_csv(DATA_DIR / "train_FD001.txt", sep=r"\s+", header=None, names=columns)
test_raw = pd.read_csv(DATA_DIR / "test_FD001.txt", sep=r"\s+", header=None, names=columns)
true_rul = pd.read_csv(DATA_DIR / "RUL_FD001.txt", sep=r"\s+", header=None, names=["RUL"])

print("Training shape:", train_raw.shape, "| Engines:", train_raw.engine_id.nunique())
print("Test shape:", test_raw.shape, "| Engines:", test_raw.engine_id.nunique())
print("Official test labels:", len(true_rul))
assert train_raw.shape[1] == test_raw.shape[1] == 26
assert train_raw.engine_id.nunique() == test_raw.engine_id.nunique() == len(true_rul) == 100
assert train_raw.isna().sum().sum() == test_raw.isna().sum().sum() == 0

## 2. RUL target and data integrity

For training engine $i$ observed at cycle $t$, the target is $RUL_i(t)=T_i-t$, where $T_i$ is its **known run-to-failure endpoint**. We use `max_cycle` and `RUL` only to construct the training targets and evaluate predictions, **never as model input features**. At deployment/test time, an engine's eventual failure cycle is unknown.

In [ ]:
df = train_raw.sort_values(["engine_id", "cycle"]).copy()
df["max_cycle"] = df.groupby("engine_id")["cycle"].transform("max")
df["RUL"] = df["max_cycle"] - df["cycle"]
final_rows = df.groupby("engine_id").tail(1)
assert (final_rows["RUL"] == 0).all()
print("Training RUL range:", (int(df.RUL.min()), int(df.RUL.max())))
print("Engines ending at RUL zero:", len(final_rows))
display(df[["engine_id", "cycle", "RUL"]].head())

## 3. Exploratory sensor behaviour

We inspect several candidate sensors and their behaviour over Engine 1's lifetime. These plots are descriptive, and pooled correlations across repeated cycle observations should **not** be interpreted as independent evidence or causal associations. Feature exploration and the reported validation experiments are exploratory; a more rigorous study would choose features using training engines only.

In [ ]:
sensor_features = ["sensor_2", "sensor_3", "sensor_4", "sensor_11"]
engine_1 = df.loc[df.engine_id == 1]
fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex=True)
for ax, sensor in zip(axes.flat, sensor_features):
    ax.plot(engine_1.cycle, engine_1[sensor], linewidth=1)
    ax.set_title(sensor)
    ax.set_xlabel("Operating cycle")
    ax.grid(alpha=0.25)
fig.suptitle("FD001 training Engine 1: selected sensors")
plt.tight_layout()
plt.show()
print("Pooled sensor/RUL correlations (descriptive):")
display(df[sensor_features + ["RUL"]].corr()["RUL"].drop("RUL").to_frame("Pearson_r").round(3))

## 4. Engine-disjoint development split

We hold out 20 complete **engines**, not random rows, for validation. This avoids mixing correlated observations from the same engine between training and validation. With only one split, validation estimates have uncertainty and may change with the sampled engines.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(splitter.split(df, groups=df["engine_id"]))
train_ids = set(df.iloc[train_idx].engine_id)
val_ids = set(df.iloc[val_idx].engine_id)
assert train_ids.isdisjoint(val_ids)
assert len(train_ids) == 80 and len(val_ids) == 20
print("Training engines:", len(train_ids), "| Validation engines:", len(val_ids))
print("Shared engine IDs:", len(train_ids.intersection(val_ids)))

## 5. Causal temporal feature engineering

For Sensor 11 we construct a five-observation trailing mean and a four-cycle difference ($S_{11}(t)-S_{11}(t-4)$). The first four differences are filled with zero as an initialization convention. Features are computed **within each engine** and never use later sensor readings.

We apply the **same transformation independently** to train and test histories. The test prediction uses only each engine's final observed row; no future sensor data are available.

In [ ]:
def add_temporal_features(frame):
    out = frame.sort_values(["engine_id", "cycle"]).copy()
    sensor_group = out.groupby("engine_id")["sensor_11"]
    out["sensor_11_roll5"] = sensor_group.transform(
        lambda x: x.rolling(window=5, min_periods=1).mean())
    out["sensor_11_change5"] = (out["sensor_11"] - sensor_group.shift(4)).fillna(0)
    return out

df = add_temporal_features(df)
test_df = add_temporal_features(test_raw)
train_df = df.loc[df.engine_id.isin(train_ids)].copy()
val_df = df.loc[df.engine_id.isin(val_ids)].copy()

rolling_features = sensor_features + ["sensor_11_roll5"]
final_features = rolling_features + ["sensor_11_change5"]
assert not df[final_features].isna().any().any()
assert not test_df[final_features].isna().any().any()
display(df.loc[df.engine_id == 1, ["cycle", "sensor_11", "sensor_11_roll5", "sensor_11_change5"]].head(10))

In [ ]:
engine_1 = df.loc[df.engine_id == 1]
plt.figure(figsize=(10, 4))
plt.plot(engine_1.cycle, engine_1.sensor_11, label="Sensor 11", alpha=0.45)
plt.plot(engine_1.cycle, engine_1.sensor_11_roll5, label="Trailing 5-cycle mean", linewidth=2)
plt.xlabel("Operating cycle")
plt.ylabel("Sensor 11")
plt.title("Current sensor reading vs. recent-history summary")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 6. Validation model comparison

We use identical engine-disjoint training and validation groups for all models. Four instantaneous sensors form the reference feature set; each subsequent Random Forest adds one temporal feature. Random Forest hyperparameters remain fixed (`n_estimators=100`, `max_depth=10`, `min_samples_leaf=5`, `random_state=42`). The following metrics are **computed by running the models**, not inserted as prewritten results.

MAE and RMSE are measured in operating cycles. $R^2$ is a variance-relative squared-error metric, **not prediction accuracy**. Since these metrics pool all validation cycle rows, engines with more observations contribute more to the scores.

In [ ]:
RF_CONFIG = dict(n_estimators=100, max_depth=10,
                 min_samples_leaf=5, random_state=42, n_jobs=-1)

model_specs = [
    ("Linear Regression — 4 sensors", LinearRegression(), sensor_features),
    ("Random Forest — 4 sensors", RandomForestRegressor(**RF_CONFIG), sensor_features),
    ("RF + rolling mean", RandomForestRegressor(**RF_CONFIG), rolling_features),
    ("RF + rolling mean + change", RandomForestRegressor(**RF_CONFIG), final_features),
]
validation_results = []
for name, estimator, used_features in model_specs:
    estimator.fit(train_df[used_features], train_df["RUL"])
    predictions = estimator.predict(val_df[used_features])
    validation_results.append({
        "Model": name,
        "MAE": mean_absolute_error(val_df.RUL, predictions),
        "RMSE": np.sqrt(mean_squared_error(val_df.RUL, predictions)),
        "R2": r2_score(val_df.RUL, predictions),
    })
validation_table = pd.DataFrame(validation_results)
display(validation_table.round(3))

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(validation_table.Model, validation_table.MAE)
ax.invert_yaxis()
ax.set_xlabel("Validation MAE (cycles; lower is better)")
ax.set_title("FD001 model and feature comparison")
plt.tight_layout()
plt.show()

## 7. Retrospective validation diagnostics

The selected six-feature configuration can have different error patterns at different points in an engine's lifetime. For a transparent illustration, we inspect its predictions at **cycle 100** for the 20 validation engines. This is a predetermined observation time, unlike selecting rows by their true (future-known) RUL. This is a *diagnostic*, not a separate or independent test.

In [ ]:
selected_rf = RandomForestRegressor(**RF_CONFIG)
selected_rf.fit(train_df[final_features], train_df.RUL)
cycle100 = val_df.loc[val_df.cycle == 100, ["engine_id", "cycle", "RUL"]].copy()
cycle100["Predicted_RUL"] = selected_rf.predict(val_df.loc[val_df.cycle == 100, final_features])
cycle100["Error"] = cycle100.Predicted_RUL - cycle100.RUL
assert len(cycle100) == 20
print("Cycle 100 | Engines:", len(cycle100))
print("MAE:", round(cycle100.Error.abs().mean(), 3))
print("Bias (predicted − actual):", round(cycle100.Error.mean(), 3))
print("Overestimation rate (%):", round(100 * (cycle100.Error > 0).mean(), 2))
display(cycle100.head(10).round(2))

## 8. Frozen final model and official FD001 test

After exploring validation results, we freeze the **six-feature Random Forest** and retrain it on all 100 FD001 training engines. The official test file has 100 partially observed engines; the ground-truth file provides one true RUL per engine at its final observation. Each engine therefore contributes **one** test prediction.

**Do not tune the model based on the following official test scores.** The all-cycle development metrics and one-endpoint-per-engine test metrics measure different prediction sampling schemes.

In [ ]:
final_rf = RandomForestRegressor(**RF_CONFIG)
final_rf.fit(df[final_features], df.RUL)

test_endpoints = (test_df.groupby("engine_id").tail(1)
                  .sort_values("engine_id").copy())
assert test_endpoints.engine_id.tolist() == list(range(1, 101))
assert len(true_rul) == len(test_endpoints) == 100

pred_test = final_rf.predict(test_endpoints[final_features])
y_test = true_rul.RUL.to_numpy()
comparison = test_endpoints[["engine_id", "cycle"]].copy()
comparison["Actual_RUL"] = y_test
comparison["Predicted_RUL"] = pred_test
comparison["Error"] = pred_test - y_test

metrics = {
    "Test engines": len(comparison),
    "MAE (cycles)": mean_absolute_error(y_test, pred_test),
    "RMSE (cycles)": np.sqrt(mean_squared_error(y_test, pred_test)),
    "R2": r2_score(y_test, pred_test),
    "Mean bias (cycles)": comparison.Error.mean(),
    "Overestimation rate (%)": 100 * (comparison.Error > 0).mean(),
    "Worst overestimation (cycles)": comparison.Error.max(),
    "Worst underestimation (cycles)": comparison.Error.min(),
}
display(pd.Series(metrics, name="Official FD001 test").round(3).to_frame())
display(comparison.head(10).round(2))

## 9. Error direction and reliability implications

**Signed error = predicted RUL − actual RUL.** Positive values overestimate remaining life; negative values underestimate it. Overestimation can be relevant to delayed-maintenance risk, but this analysis does not establish a maintenance policy, calibrated uncertainty, or operational safety guarantee.

In [ ]:
print("Five largest overestimations:")
display(comparison.nlargest(5, "Error")[["engine_id", "cycle", "Actual_RUL", "Predicted_RUL", "Error"]].round(2))

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(comparison.Actual_RUL, comparison.Predicted_RUL, alpha=0.65)
limit = max(comparison.Actual_RUL.max(), comparison.Predicted_RUL.max())
ax.plot([0, limit], [0, limit], "r--", label="Perfect prediction")
ax.set_xlabel("Actual RUL (cycles)")
ax.set_ylabel("Predicted RUL (cycles)")
ax.set_title("NASA C-MAPSS FD001 — official test")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

## 10. Conclusions, limitations and next steps

On the **single engine-disjoint validation split**, adding a trailing rolling mean and recent-change feature improved Random Forest's cycle-level MAE relative to using the four instantaneous sensor readings alone. On the **official FD001 test**, the frozen six-feature Random Forest produced substantial positive RUL prediction bias and frequent overestimation. The exact values above are computed directly from notebook execution.

**Limitations:** FD001 is simulated (one operating condition and one fault mode); our input features are limited; selection was exploratory with one validation split; row-weighted validation and endpoint-based test results are not directly comparable; short rolling summaries do not constitute full sequence modelling. Correlations examined during initial EDA included validation engines, so the sensor shortlist should not be presented as a strictly train-only feature-selection procedure. No prediction intervals, uncertainty calibration or operational risk guarantees were established.

**Future directions:** Train-only feature selection, repeated engine-group cross-validation, stronger conventional baselines, a sequence model (e.g. LSTM), and asymmetric error assessment. **Do not use the official test labels for further model tuning.**

**Reproducibility:** Run all cells top-to-bottom after attaching `behrad3d/nasa-cmaps` in Kaggle. The dataset is referenced by its Kaggle path and is **not redistributed** with this notebook.